In [ ]:
# =============================================================================
# Part 4 : 구매 시점 예측 (Timing Prediction)
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import gc

warnings.filterwarnings('ignore')

def set_korean_font():
    for font in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
        if font in {f.name for f in fm.fontManager.ttflist}:
            plt.rcParams['font.family'] = font
            break
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()

DATA_PATH = r'C:\Users\LG\K-Pick\data\prep\k-pick_total_v3.csv'

print("=" * 60)
print("Part 4 : 구매 시점 예측 (Timing Prediction)")
print("=" * 60)
print("차별화: 재구매 여부(0/1) → 언제 구매할지 4구간 다중 분류")
print("  긴급(≤7일) / 단기(8-14일) / 중기(15-30일) / 장기(>30일)")

# =============================================================================
# 1. 데이터 로드
# =============================================================================
all_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

candidate_feats = [
    'user_avg_days_between', 'user_std_days_between',
    'user_total_orders', 'user_reorder_ratio',
    'up_recency', 'up_order_count', 'up_reorder_rate',
    'up_user_product_preference', 'up_first_cart_rate',
    'prod_reorder_rate', 'prod_avg_cart_pos', 'prod_reorder_times',
    'order_hour_of_day', 'order_dow', 'favorite_dow', 'favorite_hour',
    'aisle_id', 'department_id',
]
target_col  = 'days_since_prior_order'
load_cols   = [c for c in candidate_feats + [target_col] if c in all_cols]

dtype_map = {c: 'float32' for c in load_cols}
timing_df = pd.read_csv(DATA_PATH, usecols=load_cols, dtype=dtype_map, low_memory=True)

timing_df[target_col] = timing_df[target_col].fillna(30.0)
print(f"\n로드 완료 shape: {timing_df.shape}")

# =============================================================================
# 2. 타겟 생성 : 구매 주기 구간 (4-class)
# =============================================================================
BINS          = [-1, 7, 14, 30, 999]
BUCKET_LABELS = ['긴급(≤7일)', '단기(8-14일)', '중기(15-30일)', '장기(>30일)']

timing_df['time_bucket'] = pd.cut(
    timing_df[target_col], bins=BINS, labels=[0, 1, 2, 3]
).astype('int8')

print("\n구매 시점 구간 분포:")
for i, name in enumerate(BUCKET_LABELS):
    cnt = (timing_df['time_bucket'] == i).sum()
    print(f"  {name}: {cnt:,}건 ({cnt/len(timing_df)*100:.1f}%)")

# ─── 핵심 수정: 실제로 존재하는 클래스만 labels/target_names에 반영 ───
existing_classes = sorted(timing_df['time_bucket'].unique().tolist())
existing_labels  = [BUCKET_LABELS[i] for i in existing_classes]
print(f"\n실제 존재 클래스: {existing_classes} → {existing_labels}")

feat_cols_tim = [c for c in load_cols if c != target_col]
X_tim = timing_df[feat_cols_tim].fillna(-1).values.astype(np.float32)
y_tim = timing_df['time_bucket'].values.astype(np.int8)
del timing_df; gc.collect()

# =============================================================================
# 3. Train / Val / Test 분할
# =============================================================================
from sklearn.model_selection import train_test_split

X_tt, X_te, y_tt, y_te = train_test_split(
    X_tim, y_tim, test_size=0.1, random_state=42, stratify=y_tim
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tt, y_tt, test_size=float(2)/9, random_state=42, stratify=y_tt
)
del X_tim, X_tt, y_tt; gc.collect()

print(f"\nTrain {len(y_tr):,} / Val {len(y_va):,} / Test {len(y_te):,}")

# 분할 후 테스트 세트 클래스 확인
te_classes = sorted(np.unique(y_te).tolist())
te_labels  = [BUCKET_LABELS[i] for i in te_classes]
print(f"테스트 세트 실제 클래스: {te_classes} → {te_labels}")

# =============================================================================
# 4. XGBoost 멀티클래스 분류
# =============================================================================
from xgboost import XGBClassifier

# num_class는 항상 4로 고정 (모델 구조 일관성)
print("\nXGBoost 구매 시점 예측 학습 중...")
tim_model = XGBClassifier(
    n_estimators          = 400,
    learning_rate         = 0.05,
    max_depth             = 5,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    objective             = 'multi:softprob',
    num_class             = 4,
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 25,
    tree_method           = 'hist',
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 0,
)
tim_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
print(f"조기 종료 best round: {tim_model.best_iteration}")

# =============================================================================
# 5. 성능 평가
# =============================================================================
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

y_pred_tim = tim_model.predict(X_te)
y_prob_tim = tim_model.predict_proba(X_te)   # shape (N, 4)

print(f"\n정확도: {accuracy_score(y_te, y_pred_tim):.4f}")
print("\n분류 리포트 (구매 시점 구간별):")

# ─── 핵심 수정: labels + target_names 동적 지정 ───
print(classification_report(
    y_te, y_pred_tim,
    labels      = te_classes,   # 테스트 세트에 실제 존재하는 클래스
    target_names= te_labels,    # 해당 클래스명
    zero_division= 0
))

# =============================================================================
# 6. 시각화
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
COLORS_TIM = ['#E24B4A', '#F5A623', '#1D9E75', '#534AB7']

# 6-A. 구간별 예측 확률 분포 (존재하는 클래스만)
ax0 = axes[0]
for i, name in zip(te_classes, te_labels):
    ax0.hist(
        y_prob_tim[:, i], bins=40,
        alpha=0.6, color=COLORS_TIM[i],
        label=name, density=True
    )
ax0.set(xlabel='예측 확률', ylabel='밀도', title='구매 시점 구간별 예측 확률 분포')
ax0.legend(fontsize=8)

# 6-B. Confusion Matrix (존재하는 클래스만)
cm   = confusion_matrix(y_te, y_pred_tim, labels=te_classes)
disp = ConfusionMatrixDisplay(cm, display_labels=te_labels)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (구매 시점 구간)')
axes[1].tick_params(axis='x', labelrotation=15)

# 6-C. 변수 중요도 Top 15
imp_df = (
    pd.DataFrame({'feature': feat_cols_tim,
                  'importance': tim_model.feature_importances_})
    .sort_values('importance', ascending=True)
    .tail(15)
)
axes[2].barh(imp_df['feature'], imp_df['importance'], color='#534AB7', alpha=0.85)
axes[2].set(xlabel='Importance', title='구매 시점 예측 — 변수 중요도 Top 15')

plt.tight_layout()
plt.show()

# =============================================================================
# 7. 비즈니스 인사이트
# =============================================================================
print("\n" + "=" * 60)
print("비즈니스 인사이트: 구매 시점 예측 활용")
print("=" * 60)

imminent_prob = y_prob_tim[:, 0]   # 긴급(≤7일) 구간 확률
for threshold in [0.4, 0.5, 0.6]:
    mask = imminent_prob >= threshold
    print(f"  7일 이내 구매 확률 ≥ {threshold:.1f} → {mask.sum():,}건 ({mask.mean()*100:.1f}%)")

print("""
  활용 방안:
    1) 긴급(≤7일) 고객 → 즉시 쿠폰 / 푸시 알림 발송
    2) 단기(8-14일) 고객 → 재고 소진 임박 메시지로 구매 촉진
    3) 장기(>30일) 고객 → 장기 구독 할인 상품 노출
    4) 기존 재구매 예측과 AND 조합 → "곧 재구매할 고객"에 집중 마케팅
""")
print("Part 4 완료 ✓  →  tim_model, y_prob_tim, feat_cols_tim 유지")